# Modelo Preditivo
### Grupo:
* Gabriel Koji Kondo;
* João Vitor Vargas Pereira;
* Rafael Barreto da Cruz;
* Raissa Arruda Casale;
* Ryan Lionel de Souza.

O objetivo deste notebook é prever o próximo título a ser assitido por um usuário da Netflix, baseado em informações sobre o conteúdo consumido pela conta

## 0. Importações

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
import shap

## 1. Carregamento de dados

In [2]:
df_api = pd.read_csv('../dados-transformados/api.csv')
df_viewing = pd.read_csv('../dados-transformados/Viewing_en.csv')
df_viewing[['Title_Name', 'Temporada', 'Episodio']] = df_viewing['Title'].str.split(': ',n=2, expand=True)
df_viewing.drop('Title', axis=1, inplace=True)
df_viewing["Title_Name"] = df_viewing["Title_Name"].str.replace(r"\(.*?\)", "", regex=True)

## 2. Pré Processamento

In [6]:
# titulo como chave 
df = df_viewing.merge(df_api, on="Title_Name", how="left")
df = df.sort_values(["Profile Name", "Start Time"])
df["dia_semana"] = df["Start Time"].dt.dayofweek
df["hora"] = df["Start Time"].dt.hour
generos = df["genres"].unique()

for g in generos:
    dummies = pd.get_dummies(df["genero"], prefix="flag")
    df = pd.concat([df, dummies], axis=1)
    
    props = dummies.groupby(df["user_id"]).transform("mean")
    props = props.add_prefix("prop_")
    df = pd.concat([df, props], axis=1)

#df["proximo_genero"] = df.groupby("Profile Name")["genres"].shift(-1)

,Duration,Start Time,Profile Name,Country,Bookmark,Latest Bookmark,Supplemental Video Type,Attributes,Device Type,Title_Name,...,interestingMoment._342x192.webp.value.image_key,summary.type,summary.unifiedEntityId,summary.id,summary.isOriginal,summary.liveEvent.hasLiveEvent,summary.idx,summary.episode,summary.season,summary.isPlayable
0,00:00:05,2026-01-30 14:59:45,Extra,BR (Brazil),00:56:20,00:56:20,NaN,NaN,DefaultWidevineAndroidPhone,Bridgerton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00:05:47,2026-01-30 14:39:46,Extra,BR (Brazil),00:54:40,Not latest view,NaN,NaN,DefaultWidevineAndroidPhone,Bridgerton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00:40:58,2026-01-30 09:18:13,Extra,BR (Brazil),00:40:09,Not latest view,NaN,NaN,DefaultWidevineAndroidPhone,Bridgerton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00:00:38,2026-01-30 08:40:42,Extra,BR (Brazil),00:25:59,00:25:59,NaN,NaN,DefaultWidevineAndroidPhone,Bridgerton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00:06:37,2026-01-30 01:26:09,Extra,BR (Brazil),00:20:14,00:20:14,NaN,NaN,Chrome OS (Cadmium),Dr. House,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
